# 흐르는-직선(flow-line) 시간지연 모델

1차원 구성(평행이동하는 직선 위를 달리는 상대론적 점)에서 출발해, 그 진행거리 비를 시간 지연 비 `t(r)/t(∞)` 와 동일시했을 때 나오는 직선의 속력 `w(r)` 를 구하고 Schwarzschild 해 및 Newton 중력과 비교한다.

이 노트북은 **저장소를 클론하지 않는다.** `flowline` 패키지 소스를 아래 셀에 그대로 담아 두었으므로, 저장소가 비공개여도 코랩 인증 없이 항상 똑같이 동작한다 (원래 git clone 방식은 코랩 실행 환경이 비공개 저장소에 접근할 인증 수단이 없어 `CalledProcessError: exit status 128` 로 실패했다).

자세한 설계 논의와 측정 규약 4종(A / B1 / B2 / C)의 선택 근거는 저장소 [README](https://github.com/yeongkim0814-svg/Repository-for-Claude/blob/claude/time-delay-model-design-f8jlrj/README.md) 참고 (저장소 접근 권한이 있는 경우).

## 1. 준비

- 그림의 축·범례 라벨이 한글이라 **한글 폰트**가 필요하다. 로그축 눈금에 쓰이는 마이너스 글리프(`U+2212`)가 나눔 계열엔 없어 Noto CJK 를 설치한다.
- Colab 은 `numpy`/`matplotlib` 가 이미 있으므로 `mpmath` 만 추가로 설치한다 (정합성 검사가 60자리 고정밀 계산을 쓰기 때문에 필요하다).

In [ ]:
# 한글 폰트 (Colab 은 root 로 실행되므로 apt-get 이 바로 된다)

!apt-get -qq update && apt-get -qq install -y fonts-noto-cjk > /dev/null

!pip install -q mpmath



# 방금 설치한 폰트를 matplotlib 가 다시 찾도록 캐시를 비운다.

# (flowline.plots 는 뒤에서 import 될 때 폰트 목록을 스스로 훑어

#  한글+마이너스 글리프를 모두 가진 폰트를 고른다.)

!rm -rf ~/.cache/matplotlib

## 2. flowline 패키지 내장

아래 셀들은 `%%writefile` 로 `flowline` 패키지와 테스트를 `/content/pkg` 아래에 그대로 풀어 놓는다 — 네트워크도, GitHub 인증도 필요 없다. 각 셀의 내용은 저장소의 같은 이름 파일과 **바이트 단위로 동일**하다 (이 노트북을 만든 스크립트가 저장소 파일을 그대로 읽어 넣었다).

In [ ]:
import pathlib



PKG_DIR = pathlib.Path('/content/pkg')

(PKG_DIR / 'flowline').mkdir(parents=True, exist_ok=True)

(PKG_DIR / 'tests').mkdir(parents=True, exist_ok=True)

print('생성:', PKG_DIR)

In [ ]:
%%writefile /content/pkg/flowline/__init__.py
"""흐르는-직선(flow-line) 시간지연 모델.

1차원 구성에서 출발해 Schwarzschild 해 및 Newton 중력과 비교한다.
자세한 내용은 저장소 README.md 참고.
"""


In [ ]:
%%writefile /content/pkg/flowline/constants.py
"""물리 상수와 가상 천체 정의.

GM 값은 IAU 공칭값(nominal)을 사용한다. G 와 M 을 따로 곱하면 G 의
상대 불확도(~2e-5)가 그대로 들어오지만, GM 은 훨씬 정밀하게 알려져
있기 때문이다. 이 모델의 비교는 1e-16 수준의 잔차를 다루므로
이 구분이 실제로 의미가 있다.
"""

from dataclasses import dataclass

from mpmath import mp

# 고정밀 계산용. 지구 표면에서 r_s/r ~ 1.4e-9 이라 배정밀도로는
# 모델-이론 잔차가 반올림 잡음에 묻힌다.
mp.dps = 60

C = mp.mpf("299792458")                    # 광속 [m/s], 정의값
GM_SUN = mp.mpf("1.32712440018e20")        # 태양 중력계수 [m^3/s^2]
GM_EARTH = mp.mpf("3.986004418e14")        # 지구 중력계수 [m^3/s^2]


def schwarzschild_radius(GM):
    """r_s = 2GM/c^2."""
    return 2 * GM / C**2


@dataclass(frozen=True)
class Body:
    """가상(또는 실재) 천체의 진공 외부 한 지점.

    r 은 절대 반지름[m] 으로 주거나, r_over_rs 로 r_s 의 배수로 준다.
    """

    label: str
    GM: mp.mpf
    r_m: float | None = None
    r_over_rs: float | None = None

    @property
    def rs(self):
        return schwarzschild_radius(self.GM)

    @property
    def r(self):
        if self.r_m is not None:
            return mp.mpf(self.r_m)
        if self.r_over_rs is not None:
            return mp.mpf(self.r_over_rs) * self.rs
        raise ValueError(f"{self.label}: r_m 또는 r_over_rs 중 하나가 필요하다")

    def __post_init__(self):
        if (self.r_m is None) == (self.r_over_rs is None):
            raise ValueError("r_m 과 r_over_rs 중 정확히 하나만 지정할 것")


# 약한장 -> 강한장 순서. 모델이 어느 영역까지 버티는지 보기 위한 사다리.
BODIES = [
    Body("지구 표면 (약한장)", GM_EARTH, r_m=6.3781e6),
    Body("태양 표면", GM_SUN, r_m=6.957e8),
    Body("백색왜성 0.6 M_sun, R=9000 km", mp.mpf("0.6") * GM_SUN, r_m=9.0e6),
    Body("중성자별 1.5 M_sun, R=12 km", mp.mpf("1.5") * GM_SUN, r_m=1.2e4),
    Body("블랙홀 10 M_sun, r=100 r_s", mp.mpf("10") * GM_SUN, r_over_rs=100.0),
    Body("블랙홀 10 M_sun, r=3 r_s (ISCO)", mp.mpf("10") * GM_SUN, r_over_rs=3.0),
    Body("블랙홀 10 M_sun, r=1.01 r_s (지평선 근방)", mp.mpf("10") * GM_SUN, r_over_rs=1.01),
]


In [ ]:
%%writefile /content/pkg/flowline/kinematics.py
"""1차원 흐르는-직선(flow-line) 구성의 순수 특수상대론 부분.

구성
----
실험실계 S = 무한원의 정지 관측자.
  * 직선은 자기 자신과 평행하게, 속력 w 로 평행이동한다.
    방향은 점의 진행 방향과 반대 (= 천체 쪽, 안쪽).
  * 점은 직선의 정지계 S' 에서 직선을 따라 속력 u 로 나아간다.
  * S' 는 S 에 대해 속도 -w 로 움직인다.

"시계"는 점이 직선 눈금 위에서 나아간 거리다. 정지한 직선(w=0)에서의
같은 양과의 비가 시간 지연 비 t(r)/t(inf) 의 후보가 된다.

측정 규약
--------
"일정 시간 T 동안 나아간 거리" 는 아직 미결정 표현이다. 어느 계에서
시간을 재고 어느 계에서 거리를 재느냐에 따라 네 가지로 갈라지며,
결과가 통째로 달라진다. 네 규약을 모두 구현해 비교한다.

  A  : 실험실계 좌표에서 잰 점의 변위.
  B1 : 직선에 붙은 고유시계(출발 눈금)로 잰 경과시간 동안 점이 나아간
       눈금 거리.  <-- 채택
  B2 : 점이 도착한 사건의 직선-좌표를 읽은 것. 동시성의 상대성 때문에
       Doppler 인자가 섞여 들어온다.
  C  : B1 의 눈금 거리를 실험실계에서 (길이 수축을 포함해) 잰 것.

모든 거리는 순진한 공식이 아니라 명시적 Lorentz 부스트로부터 유도한다.
"""

from mpmath import mp

from .constants import C

CONVENTIONS = ("A", "B1", "B2", "C")


def boost(t, x, v, c=C):
    """실험실계 S 의 사건 (t, x) 를, S 에 대해 속도 v 로 움직이는 S' 로 변환."""
    gamma = 1 / mp.sqrt(1 - (v / c) ** 2)
    return gamma * (t - v * x / c**2), gamma * (x - v * t)


def point_lab_velocity(u, w, c=C):
    """직선 정지계에서 +u 로 가는 점의, 실험실계에서의 속도.

    S' 가 S 에 대해 -w 로 움직이므로 속도 덧셈은 (u - w)/(1 - u w / c^2).
    """
    return (u - w) / (1 - u * w / c**2)


def travelled_distances(u, w, T, c=C):
    """실험실계 경과시간 T 동안 점이 '나아간 거리' 를 네 규약으로 계산.

    반환값의 단위는 규약마다 다른 계에서 잰 길이지만, 항상 정지한 직선
    (w=0) 에서의 같은 양으로 나눠 쓰므로 비는 무차원이다.
    """
    v_frame = -w  # S' 의 S 에 대한 속도
    up = point_lab_velocity(u, w, c)

    # 점의 세계선: E0 = (0, 0) -> E1 = (T, up*T)
    _, x_arrival = boost(T, up * T, v_frame, c)

    # 출발 눈금에 붙어 직선과 함께 흐르는 시계: O0 = (0,0) -> O1 = (T, -w*T)
    tau_line, _ = boost(T, -w * T, v_frame, c)

    xi_marks = u * tau_line                       # 직선 눈금으로 잰 진행 거리
    contraction = mp.sqrt(1 - (w / c) ** 2)       # 1/gamma

    return {
        "A": up * T,
        "B1": xi_marks,
        "B2": x_arrival,
        "C": xi_marks * contraction,
    }


def ratio(convention, u, w, c=C, T=None):
    """움직이는 직선 / 정지한 직선 의 진행거리 비. T 에 무관하다(선형)."""
    if convention not in CONVENTIONS:
        raise ValueError(f"알 수 없는 규약: {convention!r}")
    T = mp.mpf(1) if T is None else T
    moving = travelled_distances(u, w, T, c)[convention]
    static = travelled_distances(u, mp.mpf(0), T, c)[convention]
    return moving / static


# ---------------------------------------------------------------------------
# 위 부스트에서 손으로 정리하면 나오는 닫힌 형태. 수치 검증용이자,
# 각 규약이 u 에 의존하는지 한눈에 보게 해 준다.
#
#   A  : (1 - w/u) / (1 - u w / c^2)                        u 의존
#   B1 : sqrt(1 - w^2/c^2)                                  u 무관  <-- 채택
#   B2 : sqrt(1 - w^2/c^2) / (1 - u w / c^2)                u 의존
#   C  : 1 - w^2/c^2                                        u 무관
# ---------------------------------------------------------------------------


def ratio_closed_form(convention, u, w, c=C):
    beta = w / c
    if convention == "A":
        return (1 - w / u) / (1 - u * w / c**2)
    if convention == "B1":
        return mp.sqrt(1 - beta**2)
    if convention == "B2":
        return mp.sqrt(1 - beta**2) / (1 - u * w / c**2)
    if convention == "C":
        return 1 - beta**2
    raise ValueError(f"알 수 없는 규약: {convention!r}")


class DegenerateInversion(Exception):
    """규약이 목표 비를 결정하지 못할 때 (예: 규약 A + u=c)."""


def solve_w(convention, target_ratio, u, c=C):
    """진행거리 비가 target_ratio 가 되게 하는 직선의 속력 w 를 구한다."""
    R = mp.mpf(target_ratio)

    if convention == "B1":
        return c * mp.sqrt(1 - R**2)
    if convention == "C":
        return c * mp.sqrt(1 - R)
    if convention == "A":
        # u=c 에서 순방향 사상이 상수 1 이 되어 역산이 불가능하다. 대수적으로
        # 정리하면 w = c 라는 근이 나오지만, 그 값을 순방향에 넣으면 0/0 이라
        # 실제 해가 아니다 (허근). 목표 비를 담아내지 못한다는 뜻.
        if mp.almosteq(u, c, rel_eps=mp.mpf(10) ** (-mp.dps + 5)):
            raise DegenerateInversion(
                "규약 A 는 u=c 에서 퇴화한다: 어떤 w<c 를 넣어도 비가 1 이므로 "
                "순방향 사상이 상수이고 역산이 성립하지 않는다."
            )
        return u * (1 - R) / (1 - R * u**2 / c**2)
    if convention == "B2":
        # B2 의 비는 beta=0 에서 1, beta=u/c 에서 최대, 이후 beta->1 에서 0 으로
        # 떨어진다. 단조가 아니므로 초기추정값 뉴턴법은 복소근으로 새어나간다.
        # 감소 구간 [u/c, 1) 을 잡아 이분법으로 푼다.
        if mp.almosteq(u, c, rel_eps=mp.mpf(10) ** (-mp.dps + 5)):
            raise DegenerateInversion(
                "규약 B2 는 u=c 에서 해가 없다: 비가 sqrt((1+b)/(1-b)) >= 1 이라 "
                "1 보다 작은 시간 지연 비를 결코 만들 수 없다."
            )
        lo, hi = u / c, 1 - mp.mpf(10) ** (-mp.dps // 2)
        f = lambda beta: ratio_closed_form("B2", u, beta * c, c) - R
        return c * mp.findroot(f, (lo, hi), solver="bisect", tol=mp.mpf(10) ** (-mp.dps + 5))
    raise ValueError(f"알 수 없는 규약: {convention!r}")


In [ ]:
%%writefile /content/pkg/flowline/theory.py
"""비교 대상이 되는 기존 이론들 (Schwarzschild, Newton)."""

from mpmath import mp

from .constants import C


def schwarzschild_ratio(r, rs):
    """정지 시계의 고유시간 / 무한원 좌표시간 = sqrt(1 - r_s/r).

    이것이 t(r)/t(inf) 이며, 모델이 맞춰야 할 목표값이다.
    """
    return mp.sqrt(1 - rs / r)


def newton_potential(r, GM):
    return -GM / r


def newton_field(r, GM):
    """부호 있는 반경 성분. 안쪽으로 당기므로 음수."""
    return -GM / r**2


def escape_speed(r, GM):
    return mp.sqrt(2 * GM / r)


def weak_field_ratio(r, GM, c=C):
    """교과서 1차 근사 1 + Phi/c^2."""
    return 1 + newton_potential(r, GM) / c**2


In [ ]:
%%writefile /content/pkg/flowline/model.py
"""흐르는-직선 모델의 순방향(예측) 정식화.

역방향(교정)과 순방향(예측)을 분리하는 것이 이 파일의 요점이다.

역방향 -- calibrate_flow_speed()
    Schwarzschild 의 t(r)/t(inf) 를 목표로 놓고 직선의 속력 w(r) 를
    역산한다. "내 모델이 맞으려면 직선이 얼마나 빨라야 하는가?"

순방향 -- flow_speed() 이하
    1차원 구성만으로는 w 의 r 의존성이 나오지 않는다. 그것은 3차원
    닫힘 조건에서 온다:

        진공에서  laplacian( -w^2/2 ) = 0,   w(inf) = 0
        =>  w^2 = 2GM/r  =>  w(r) = sqrt(2GM/r)   (= 탈출속도)

    이 w(r) 를 모델의 공준으로 놓으면 Phi 와 g 가 따라 나온다:

        Phi(r) = -w^2/2            포텐셜 = 흐름의 단위질량당 운동에너지의 음수
        g(r)   = w dw/dr           중력장 = 흐름장의 이류 가속도 (Dw/Dt)

    이 그림은 Painleve-Gullstrand 좌표계 / river model 과 같은 구조다.
"""

from mpmath import mp

from .constants import C
from .kinematics import solve_w
from .theory import schwarzschild_ratio


def calibrate_flow_speed(r, rs, convention="B1", u=None, c=C):
    """Schwarzschild 시간 지연 비를 재현하도록 w(r) 를 역산한다."""
    u = c if u is None else u
    return solve_w(convention, schwarzschild_ratio(r, rs), u, c)


# --- 순방향: 3차원 닫힘 조건이 고정해 준 흐름 프로파일 ---------------------


def flow_speed(r, GM):
    """w(r) = sqrt(2GM/r). 그 지점의 탈출속도와 같다."""
    return mp.sqrt(2 * GM / r)


def potential(r, GM):
    """Phi = -w^2/2."""
    return -flow_speed(r, GM) ** 2 / 2


def field(r, GM):
    """g = w dw/dr, 수치 미분으로 계산 (해석해를 대입하지 않는다)."""
    w = flow_speed(r, GM)
    dwdr = mp.diff(lambda rr: flow_speed(rr, GM), r)
    return w * dwdr


def time_ratio(r, GM, c=C):
    """모델이 예측하는 t(r)/t(inf) = sqrt(1 - w^2/c^2)."""
    return mp.sqrt(1 - (flow_speed(r, GM) / c) ** 2)


def horizon_radius(GM, c=C):
    """w = c 가 되는 반지름. 모델의 결과로 나오는 지평선."""
    return 2 * GM / c**2


def radial_laplacian_of_potential(r, GM):
    """진공 닫힘 조건 검사: (1/r^2) d/dr ( r^2 dPhi/dr ) 이 0 인가."""
    inner = lambda rr: rr**2 * mp.diff(lambda x: potential(x, GM), rr)
    return mp.diff(inner, r) / r**2


In [ ]:
%%writefile /content/pkg/flowline/checks.py
"""모델의 정합성 검사. 각 검사는 (이름, 통과여부, 설명) 을 돌려준다."""

from mpmath import mp

from .constants import BODIES, C, GM_SUN
from . import kinematics as K
from . import model, theory

TOL = mp.mpf(10) ** (-mp.dps + 8)   # 반올림 여유를 둔 '정확히 같다' 기준


def _rel(a, b):
    """상대 오차. b=0 이면 절대 오차."""
    return abs(a - b) / abs(b) if b != 0 else abs(a - b)


def _max_over_bodies(residual_fn, bodies=BODIES):
    """bodies 를 순회하며 residual_fn(body) 의 최댓값과 그 위치를 돌려준다.

    check_calibrated_w_is_escape_speed / check_forward_ratio_matches_schwarzschild /
    check_vacuum_closure 가 공유하던 'worst, where 갱신 루프' 를 한 곳으로 모은 것.
    """
    worst, where = mp.mpf(0), ""
    for b in bodies:
        d = residual_fn(b)
        if d > worst:
            worst, where = d, b.label
    return worst, where


def check_boost_vs_closed_form():
    """네 규약의 닫힌 형태가 명시적 Lorentz 부스트와 일치하는가."""
    worst, where = mp.mpf(0), ""
    for uf in ("0.1", "0.5", "0.9", "1.0"):
        for wf in ("0.05", "0.3", "0.7", "0.95"):
            u, w = mp.mpf(uf) * C, mp.mpf(wf) * C
            for conv in K.CONVENTIONS:
                d = _rel(K.ratio(conv, u, w), K.ratio_closed_form(conv, u, w))
                if d > worst:
                    worst, where = d, f"{conv}, u={uf}c, w={wf}c"
    return ("부스트 유도 == 닫힌 형태", worst < TOL,
            f"최대 상대오차 {mp.nstr(worst, 3)} ({where})")


def check_u_independence():
    """채택 규약 B1 은 점의 속력 u 에 무관해야 한다.

    시간 지연은 '무엇으로 재느냐' 에 의존하면 안 되므로, 이것이
    규약 선택의 핵심 판정 기준이다. A 와 B2 는 여기서 탈락한다.
    """
    w = mp.mpf("0.6") * C
    us = [mp.mpf(f) * C for f in ("0.1", "0.3", "0.5", "0.9", "1.0")]
    spread = {}
    for conv in K.CONVENTIONS:
        vals = []
        for u in us:
            try:
                vals.append(K.ratio_closed_form(conv, u, w))
            except ZeroDivisionError:
                vals.append(mp.inf)
        spread[conv] = max(vals) - min(vals)
    ok = spread["B1"] < TOL and spread["C"] < TOL and spread["A"] > 1e-3 and spread["B2"] > 1e-3
    detail = ", ".join(f"{c}: 폭 {mp.nstr(spread[c], 4)}" for c in K.CONVENTIONS)
    return ("u-독립성 (B1, C 만 통과)", ok, detail + "  [w=0.6c, u=0.1c~c]")


def check_convention_A_degenerate():
    """규약 A 는 u=c 에서 어떤 w 를 넣어도 비가 1 -> 정보가 없다."""
    vals = [K.ratio_closed_form("A", C, mp.mpf(f) * C) for f in ("0.1", "0.5", "0.9")]
    flat = all(_rel(v, 1) < TOL for v in vals)
    try:
        K.solve_w("A", mp.mpf("0.5"), C)
        raised = False
    except K.DegenerateInversion:
        raised = True
    return ("규약 A 는 u=c 에서 퇴화", flat and raised,
            "w=0.1c/0.5c/0.9c 모두 비=1, 역산은 DegenerateInversion 을 던짐")


def check_B2_is_doppler():
    """B2 는 비가 1 을 넘어 '시간이 빨리 간다' 가 되어 버린다."""
    w = mp.mpf("0.6") * C
    val = K.ratio_closed_form("B2", C, w)
    expected = mp.sqrt((1 + w / C) / (1 - w / C))     # 상대론적 Doppler 인자
    return ("B2 = Doppler 인자 (기각 근거)", val > 1 and _rel(val, expected) < TOL,
            f"u=c, w=0.6c 에서 비 = {mp.nstr(val, 10)} > 1, "
            f"sqrt((1+b)/(1-b)) 와 일치")


def check_calibrated_w_is_escape_speed():
    """B1 로 역산한 w 가 정확히 탈출속도인가 (약한장 근사가 아니라 전 영역)."""
    worst, where = _max_over_bodies(
        lambda b: _rel(model.calibrate_flow_speed(b.r, b.rs, "B1"),
                       theory.escape_speed(b.r, b.GM)))
    return ("역산 w(r) == 탈출속도 sqrt(2GM/r)", worst < TOL,
            f"최대 상대오차 {mp.nstr(worst, 3)} ({where})")


def check_forward_ratio_matches_schwarzschild():
    """순방향 예측이 Schwarzschild 와 정확히 일치하는가."""
    worst, where = _max_over_bodies(
        lambda b: _rel(model.time_ratio(b.r, b.GM), theory.schwarzschild_ratio(b.r, b.rs)))
    return ("순방향 t(r)/t(inf) == Schwarzschild", worst < TOL,
            f"최대 상대오차 {mp.nstr(worst, 3)} ({where})")


def check_potential_and_field():
    """Phi = -w^2/2 와 g = w dw/dr 가 Newton 값과 맞는가."""
    worst_phi, _ = _max_over_bodies(
        lambda b: _rel(model.potential(b.r, b.GM), theory.newton_potential(b.r, b.GM)))
    worst_g, _ = _max_over_bodies(
        lambda b: _rel(model.field(b.r, b.GM), theory.newton_field(b.r, b.GM)))
    return ("Phi = -w^2/2, g = w dw/dr 가 Newton 과 일치",
            worst_phi < TOL and worst_g < TOL,
            f"Phi 최대오차 {mp.nstr(worst_phi, 3)}, g 최대오차 {mp.nstr(worst_g, 3)} "
            "(g 는 수치 미분)")


def check_vacuum_closure():
    """3차원 닫힘 조건: 진공에서 laplacian(-w^2/2) = 0.

    1차원 구성이 스스로 낳지 못하는 유일한 재료가 이것이고,
    이것이 w 의 r^(-1/2) 의존성을 고정한다.
    """
    def normalized_residual(b):
        lap = model.radial_laplacian_of_potential(b.r, b.GM)
        scale = abs(theory.newton_field(b.r, b.GM)) / b.r    # 같은 지점의 g/r 규모
        return abs(lap) / scale

    worst, where = _max_over_bodies(normalized_residual)
    return ("진공 닫힘 조건 laplacian(-w^2/2) = 0", worst < mp.mpf("1e-20"),
            f"최대 규격화 잔차 {mp.nstr(worst, 3)} ({where})")


def check_weak_field_coefficient():
    """약한장에서 모델 - 교과서 1차식 의 잔차가 -(1/8)(r_s/r)^2 인가.

    1차 항이 맞는 건 당연하고, 2차 항의 계수까지 맞는지가 진짜 검사다.
    """
    GM = GM_SUN
    rs = 2 * GM / C**2
    rows, ok = [], True
    for e in (6, 9, 12):
        r = rs * mp.mpf(10) ** e
        eps = rs / r
        resid = model.time_ratio(r, GM) - theory.weak_field_ratio(r, GM)
        coeff = resid / eps**2
        # 다음 항이 -(1/16) eps^3 이므로 계수의 편차는 O(eps) 로 줄어들어야
        # 한다. 상수 허용오차가 아니라 이 수렴 속도 자체를 검사한다.
        dev = _rel(coeff, mp.mpf(-1) / 8)
        ok = ok and dev < 10 * eps
        rows.append(f"r/r_s=1e{e}: 계수 {mp.nstr(coeff, 12)} (편차/eps {mp.nstr(dev / eps, 4)})")
    return ("약한장 2차 계수 == -1/8 (수렴속도 O(eps) 포함)", ok,
            "; ".join(rows) + "  (기대 -0.125, 편차/eps -> 0.5)")


def check_horizon_emerges():
    """w = c 인 지점이 r_s 와 같은가. 가정이 아니라 결과여야 한다."""
    GM = mp.mpf(10) * GM_SUN
    r_h = model.horizon_radius(GM)
    rs = 2 * GM / C**2
    return ("지평선이 결과로 출현 (w=c <=> r=r_s)",
            _rel(r_h, rs) < TOL and _rel(model.flow_speed(r_h, GM), C) < TOL,
            f"w(r)=c 인 r = {mp.nstr(r_h, 12)} m, r_s = {mp.nstr(rs, 12)} m")


ALL_CHECKS = [
    check_boost_vs_closed_form,
    check_u_independence,
    check_convention_A_degenerate,
    check_B2_is_doppler,
    check_calibrated_w_is_escape_speed,
    check_forward_ratio_matches_schwarzschild,
    check_potential_and_field,
    check_vacuum_closure,
    check_weak_field_coefficient,
    check_horizon_emerges,
]


def run_all():
    return [fn() for fn in ALL_CHECKS]


In [ ]:
%%writefile /content/pkg/flowline/report.py
"""수치 비교표 생성."""

from mpmath import mp

from .constants import BODIES, C
from . import checks, kinematics as K, model, theory


def _fmt(x, n=6):
    return mp.nstr(mp.mpf(x), n)


def _rule(width=110, ch="-"):
    return ch * width


def _body_table(title, header_line, width, row_fn, notes, bodies=BODIES):
    """BODIES 를 한 줄씩 훑는 표의 공통 골격.

    table_time_dilation 과 table_potential_and_field 가 열 구성만 다르고
    '제목 + 헤더 + 구분선 + 행들 + 각주' 뼈대를 그대로 반복하던 것을 모았다.
    table_conventions 는 (규약 x u) 이중 순회라 구조가 달라 별도로 둔다.
    """
    lines = [title, "", header_line, _rule(width)]
    lines += [row_fn(b) for b in bodies]
    lines += [""] + list(notes)
    return "\n".join(lines)


def table_time_dilation():
    """모델 예측 t(r)/t(inf) 를 Schwarzschild 및 교과서 1차식과 비교."""
    def row(b):
        r, rs, GM = b.r, b.rs, b.GM
        w = model.flow_speed(r, GM)
        rm = model.time_ratio(r, GM)
        rt = theory.schwarzschild_ratio(r, rs)
        resid = abs(rm - rt) / rt
        weak_err = abs(theory.weak_field_ratio(r, GM) - rt) / rt
        return (f"{b.label:<40} {_fmt(r / rs, 6):>12} {_fmt(w / C, 6):>12} "
                f"{mp.nstr(rm, 18):>24} {_fmt(resid, 3):>18} {_fmt(weak_err, 3):>14}")

    return _body_table(
        "[표 1] 시간 지연 비  t(r)/t(inf)",
        f"{'천체 / 위치':<40} {'r/r_s':>12} {'w/c':>12} "
        f"{'모델 = Schwarzschild':>24} {'모델-Schw (상대)':>18} {'1차근사 오차':>14}",
        126, row,
        [
            "  * '모델'과 'Schwarzschild' 열이 하나인 이유: 두 값이 60자리 전 자리에서",
            "    같아 따로 쓸 내용이 없다. 근사가 아니라 항등이다.",
            "  * 마지막 열은 교과서 1차식 1+Phi/c^2 이 Schwarzschild 에서 벗어난 정도.",
            "    이 모델은 그 오차를 갖지 않는다.",
        ],
    )


def table_potential_and_field():
    """Phi 와 g 의 모델-Newton 대응."""
    def row(b):
        r, GM = b.r, b.GM
        phi_m, phi_n = model.potential(r, GM), theory.newton_potential(r, GM)
        g_m, g_n = model.field(r, GM), theory.newton_field(r, GM)
        return (f"{b.label:<40} {_fmt(phi_m, 10):>24} {_fmt(abs(phi_m - phi_n) / abs(phi_n), 3):>16} "
                f"{_fmt(g_m, 10):>24} {_fmt(abs(g_m - g_n) / abs(g_n), 3):>16}")

    return _body_table(
        "[표 2] 포텐셜과 중력장 -- Phi = -w^2/2,  g = w dw/dr",
        f"{'천체 / 위치':<40} {'Phi = -w^2/2 [J/kg]':>24} {'-GM/r 상대오차':>16} "
        f"{'g = w dw/dr [m/s^2]':>24} {'-GM/r^2 상대오차':>16}",
        124, row,
        [
            "  * g 는 해석해를 대입한 것이 아니라 흐름장 w(r) 를 수치 미분해 얻었다.",
            "    즉 '중력장 = 흐름의 이류 가속도 Dw/Dt' 가 실제로 성립함을 보인 것.",
        ],
    )


def table_conventions():
    """네 측정 규약이 서로 다른 w 를 낳는다는 것을 한 지점에서 보인다."""
    b = next(x for x in BODIES if "중성자별" in x.label)
    R = theory.schwarzschild_ratio(b.r, b.rs)
    lines = [
        "[표 3] 측정 규약별 역산 결과  (기준점: " + b.label + ")",
        f"        목표 비 t(r)/t(inf) = {mp.nstr(R, 16)},   탈출속도 v_esc/c = "
        f"{_fmt(theory.escape_speed(b.r, b.GM) / C, 10)}",
        "",
        f"{'규약':<6} {'설명':<44} {'u':>8} {'역산 w/c':>16} {'v_esc 대비':>14} {'판정':<10}",
        _rule(104),
    ]
    desc = {
        "A": "실험실계 좌표에서 잰 점의 변위",
        "B1": "직선 고유시계로 잰 눈금 진행거리",
        "B2": "점의 도착 사건을 직선좌표로 읽음",
        "C": "B1 의 눈금거리를 실험실계에서 읽음",
    }
    for conv in K.CONVENTIONS:
        for uf in ("0.5", "1.0"):
            u = mp.mpf(uf) * C
            try:
                w = K.solve_w(conv, R, u)
                wc, rel = _fmt(w / C, 10), _fmt(w / theory.escape_speed(b.r, b.GM), 8)
            except K.DegenerateInversion:
                wc, rel = "퇴화", "-"
            if conv == "A":
                verdict = "기각(u의존/퇴화)"
            elif conv == "B2":
                verdict = "기각(u의존)"
            elif conv == "B1":
                verdict = "채택"
            else:
                verdict = "g_tt 자체"
            lines.append(f"{conv:<6} {desc[conv]:<44} {uf + 'c':>8} {wc:>16} {rel:>14} {verdict:<10}")
    lines += [
        "",
        "  * B1 만이 u 에 무관하면서 v_esc 와 정확히 일치한다 (비율 1.0).",
        "  * C 는 비 = 1-beta^2 = g_tt 를 재므로 시간 지연 sqrt(g_tt) 가 아니다.",
        "    같은 목표 비를 넣으면 다른 w 가 나온다.",
    ]
    return "\n".join(lines)


def section_checks():
    lines = ["[검사] 정합성 검증", ""]
    n_fail = 0
    for name, ok, detail in checks.run_all():
        n_fail += not ok
        lines.append(f"  {'PASS' if ok else 'FAIL'}  {name}")
        lines.append(f"        {detail}")
    lines += ["", f"  총 {len(checks.ALL_CHECKS)}개 중 실패 {n_fail}개."]
    return "\n".join(lines), n_fail


def full_report():
    checks_text, n_fail = section_checks()
    header = [
        _rule(126, "="),
        "흐르는-직선(flow-line) 시간지연 모델  --  정합적 이론과의 비교",
        _rule(126, "="),
        "",
        "모델:  직선이 속력 w 로 (점의 진행 반대방향, 즉 천체 쪽) 평행이동하고,",
        "       점은 직선 정지계에서 속력 u 로 직선을 따라 나아간다.",
        "       점이 직선 눈금 위에서 나아간 거리가 시계 눈금이다.",
        "",
        f"계산 정밀도: {mp.dps} 자리 (mpmath)",
        "",
    ]
    return "\n".join(header) + "\n\n" + "\n\n".join([
        table_time_dilation(),
        table_potential_and_field(),
        table_conventions(),
        checks_text,
    ]), n_fail


In [ ]:
%%writefile /content/pkg/flowline/plots.py
"""그림 생성.

수치표는 mpmath 고정밀로 뽑지만, 그림은 강한장 영역을 그리므로
자릿수 소거 문제가 없다. 여기서는 float/numpy 를 쓴다.
"""

import sys

import matplotlib

# CLI(run_all.py)는 디스플레이가 없는 헤드리스 환경일 수 있어 Agg 가 필요하다.
# 반대로 Jupyter/Colab 은 노트북을 열 때 이미 IPython 이 백엔드를 골라 두므로,
# 여기서 Agg 로 덮어쓰면 노트북 안에서 plt.show() 를 쓰는 다른 셀이 깨진다.
# "이미 IPython 이 로드돼 있는가" 로 두 경우를 가른다.
if "IPython" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np


def _font_has(path, codepoints):
    """폰트가 해당 코드포인트를 실제로 담고 있는지 확인. fontTools 가 없으면 None."""
    try:
        from fontTools.ttLib import TTFont
    except ImportError:
        return None
    try:
        font = TTFont(path, fontNumber=0, lazy=True)
        cmap = set()
        for table in font["cmap"].tables:
            cmap |= set(table.cmap)
        font.close()
    except Exception:
        return None
    return all(cp in cmap for cp in codepoints)


# 한글(U+AC00) 과 마이너스(U+2212) 를 모두 담아야 한다. 후자는 로그축 눈금
# 라벨에 쓰이는데, 나눔 계열에는 없어서 눈금이 두부(tofu)가 된다.
_REQUIRED = (0xAC00, 0x2212)


def _hangul_font():
    """그림 라벨이 한글이라 한글 폰트가 필요하다. 없으면 두부가 찍힌다."""
    preferred = ("Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic",
                 "AppleGothic", "NanumGothic", "NanumBarunGothic")
    installed = {f.name: f.fname for f in fm.fontManager.ttflist}
    fallback = None
    for name in preferred:
        if name not in installed:
            continue
        ok = _font_has(installed[name], _REQUIRED)
        if ok or ok is None:
            return name
        fallback = fallback or name          # 한글은 되지만 글리프가 모자란 폰트

    import warnings

    if fallback:
        warnings.warn(
            f"{fallback} 에는 U+2212(마이너스) 글리프가 없어 로그축 눈금이 깨진다. "
            "권장: apt-get install fonts-noto-cjk",
            RuntimeWarning,
        )
        return fallback
    warnings.warn(
        "한글 폰트를 찾지 못했다. 그림의 한글 라벨이 깨진다. "
        "설치: apt-get install fonts-noto-cjk  (그 뒤 matplotlib 캐시 삭제)",
        RuntimeWarning,
    )
    return None


# dataviz 기준 팔레트 (light 모드). 계열은 고정 순서로 배정하고 순환시키지 않는다.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#dcdbd6"
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"        # 범주형 슬롯 1~3
SEQ = ["#86b6ef", "#2a78d6", "#104281"]             # 순차 램프 (u 는 연속량)

_FONT = _hangul_font()

plt.rcParams.update({
    **({"font.family": [_FONT, "DejaVu Sans"]} if _FONT else {}),
    "axes.unicode_minus": False,
    # 로그축 눈금은 mathtext 로 그려지고, mathtext 의 'default'/'normal' 슬롯은
    # 폰트셋이 아니라 본문 폰트를 쓴다. 본문 폰트에 U+2212 이 없으면 눈금이
    # 두부가 되므로, 'rm' 으로 고정해 폰트셋 쪽을 쓰게 한다.
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "rm",
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.size": 10,
    "axes.labelcolor": INK,
    "axes.edgecolor": GRID,
    "text.color": INK,
    "xtick.color": INK_2,
    "ytick.color": INK_2,
    "axes.titlesize": 11,
    "axes.titleweight": "normal",
    "lines.linewidth": 2.0,
    "legend.frameon": False,
})


def _style(ax, title, xlabel, ylabel, logx=True):
    ax.set_title(title, loc="left", color=INK, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if logx:
        ax.set_xscale("log")
    ax.grid(True, color=GRID, linewidth=0.8, alpha=0.9)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def figure_profiles(path):
    """반지름에 따른 네 프로파일. 모델(실선) vs 기존 이론(파선)."""
    x = np.logspace(0.0001, 3, 500)          # r/r_s
    beta = np.sqrt(1.0 / x)                  # w/c = sqrt(r_s/r)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))

    ax = axes[0, 0]
    ax.plot(x, np.sqrt(1 - beta**2), color=S1, label="모델  sqrt(1 - w²/c²)")
    ax.plot(x, np.sqrt(1 - 1 / x), color=S2, linestyle="--", dashes=(5, 4),
            label="Schwarzschild  sqrt(1 - r_s/r)")
    ax.plot(x, 1 - 0.5 / x, color=S3, linestyle=":", linewidth=1.8,
            label="교과서 1차 근사  1 + Φ/c²")
    _style(ax, "시간 지연 비  t(r) / t(∞)", "r / r_s", "비")
    ax.legend(loc="lower right")
    ax.annotate("두 곡선이 겹쳐 하나로 보인다\n(60자리까지 항등)",
                xy=(6, np.sqrt(1 - 1 / 6)), xytext=(20, 0.55), color=INK_2, fontsize=9,
                arrowprops=dict(arrowstyle="->", color=INK_2, linewidth=1))

    ax = axes[0, 1]
    ax.plot(x, beta, color=S1, label="직선의 속력  w/c = sqrt(r_s/r)")
    ax.axhline(1.0, color=INK_2, linewidth=1, linestyle="--", dashes=(3, 3))
    _style(ax, "직선의 평행이동 속력 (= 탈출속도)", "r / r_s", "w / c")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="upper right")
    ax.annotate("w = c  →  r = r_s\n지평선이 결과로 나옴",
                xy=(1.02, 0.99), xytext=(3, 0.62), color=INK_2, fontsize=9,
                arrowprops=dict(arrowstyle="->", color=INK_2, linewidth=1))

    ax = axes[1, 0]
    ax.plot(x, -0.5 * beta**2, color=S1, label="모델  Φ = -w²/2")
    ax.plot(x, -0.5 / x, color=S2, linestyle="--", dashes=(5, 4), label="Newton  Φ = -GM/r")
    _style(ax, "포텐셜 (c² 단위)", "r / r_s", "Φ / c²")
    ax.legend(loc="lower right")

    ax = axes[1, 1]
    ax.plot(x, -0.5 / x**2, color=S1, label="모델  g = w dw/dr")
    ax.plot(x, -0.5 / x**2, color=S2, linestyle="--", dashes=(5, 4), label="Newton  g = -GM/r²")
    _style(ax, "중력장 (c²/r_s 단위)", "r / r_s", "g r_s / c²")
    ax.set_yscale("symlog", linthresh=1e-6)
    ax.legend(loc="lower right")

    fig.suptitle("흐르는-직선 모델 vs 정합적 이론  (규약 B1)",
                 x=0.008, ha="left", fontsize=13, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    fig.savefig(path, dpi=150)
    plt.close(fig)


def figure_conventions(path):
    """측정 규약이 왜 B1 이어야 하는지를 두 패널로."""
    beta = np.linspace(0, 0.995, 500)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

    ax = axes[0]
    ax.plot(beta, np.sqrt(1 - beta**2), color=S1, label="B1  (채택) — u 무관")
    ax.plot(beta, 1 - beta**2, color=S2, linestyle="--", dashes=(5, 4),
            label="C  = g_tt, 시간지연 아님")
    ax.plot(beta, np.sqrt(1 - beta**2) / (1 - 0.5 * beta), color=S3, linestyle=":",
            linewidth=1.8, label="B2  (u=0.5c) — Doppler 오염")
    ax.axhline(1.0, color=INK_2, linewidth=1, linestyle="--", dashes=(3, 3))
    _style(ax, "규약별 진행거리 비", "w / c", "비", logx=False)
    ax.set_ylim(0, 1.42)
    ax.legend(loc="lower left")
    ax.annotate("B2 는 비 > 1\n(시간이 빨리 감)", xy=(0.5, 1.155), xytext=(0.6, 1.32),
                color=INK_2, fontsize=9, arrowprops=dict(arrowstyle="->", color=INK_2, linewidth=1))

    ax = axes[1]
    for u_over_c, color in zip((0.3, 0.6, 0.9), SEQ):
        with np.errstate(divide="ignore", invalid="ignore"):
            r = (1 - beta / u_over_c) / (1 - u_over_c * beta)
        ax.plot(beta, r, color=color, label=f"규약 A,  u = {u_over_c}c")
    ax.axhline(0.0, color=INK_2, linewidth=1)
    _style(ax, "규약 A 는 점의 속력 u 에 의존한다 (기각 근거)", "w / c", "비", logx=False)
    ax.set_ylim(-2, 1.2)
    ax.legend(loc="lower left")

    fig.suptitle("측정 규약의 선택 — 왜 B1 인가",
                 x=0.008, ha="left", fontsize=13, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(path, dpi=150)
    plt.close(fig)


def figure_weak_field(path):
    """약한장에서 모델과 교과서 1차식의 잔차가 기울기 2 로 사라지는가.

    이 곡선만은 mpmath 로 계산한다. 잔차가 eps^2/8 이라 eps < 3e-8 에서
    1e-16 아래로 내려가는데, 배정밀도로는 그 지점부터 반올림 잡음이 되어
    톱니가 생긴다. 표를 60자리로 뽑는 이유가 바로 이것이다.
    """
    from mpmath import mp

    eps_f = np.logspace(-9, -1, 160)                 # r_s / r
    resid = np.array([
        float(abs(mp.sqrt(1 - mp.mpf(e)) - (1 - mp.mpf(e) / 2))) for e in eps_f
    ])

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    ax.plot(eps_f, resid, color=S1, label="| 모델 - (1 + Φ/c²) |  (60자리 계산)")
    ax.plot(eps_f, eps_f**2 / 8, color=S2, linestyle="--", dashes=(5, 4),
            label="(1/8)(r_s/r)²  기준선")
    ax.axhline(2.2e-16, color=INK_2, linewidth=1, linestyle=":")
    ax.set_xscale("log")
    ax.set_yscale("log")
    _style(ax, "약한장 잔차 — 2차 항 계수 검증", "r_s / r", "|잔차|", logx=True)
    ax.legend(loc="upper left")
    ax.annotate("배정밀도 한계 (2.2e-16)\n이 아래는 float 로는 잡음이 된다",
                xy=(2e-9, 2.2e-16), xytext=(3e-9, 3e-13),
                color=INK_2, fontsize=9,
                arrowprops=dict(arrowstyle="->", color=INK_2, linewidth=1))
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


In [ ]:
%%writefile /content/pkg/tests/test_model.py
"""pytest 스위트. 정합성 검사 전부 + 몇 가지 성질 검사."""

import pytest
from mpmath import mp

from flowline import checks, kinematics as K, model, theory
from flowline.constants import BODIES, C, GM_SUN

TOL = mp.mpf(10) ** (-mp.dps + 8)


@pytest.mark.parametrize("check", checks.ALL_CHECKS, ids=lambda f: f.__name__)
def test_consistency_check(check):
    name, ok, detail = check()
    assert ok, f"{name}: {detail}"


@pytest.mark.parametrize("body", BODIES, ids=lambda b: b.label)
def test_forward_matches_schwarzschild(body):
    assert abs(model.time_ratio(body.r, body.GM)
               - theory.schwarzschild_ratio(body.r, body.rs)) < TOL


@pytest.mark.parametrize("body", BODIES, ids=lambda b: b.label)
def test_calibration_round_trip(body):
    """역산한 w 를 순방향에 다시 넣으면 목표 비가 돌아와야 한다."""
    target = theory.schwarzschild_ratio(body.r, body.rs)
    w = model.calibrate_flow_speed(body.r, body.rs, "B1")
    assert abs(K.ratio_closed_form("B1", C, w) - target) < TOL


@pytest.mark.parametrize("u_frac", ["0.1", "0.5", "0.9", "1.0"])
def test_B1_is_independent_of_u(u_frac):
    w = mp.mpf("0.42") * C
    u = mp.mpf(u_frac) * C
    assert abs(K.ratio("B1", u, w) - mp.sqrt(1 - (w / C) ** 2)) < TOL


def test_w_zero_gives_unit_ratio():
    """정지한 직선은 무한원과 같아야 한다 (경계조건)."""
    for conv in K.CONVENTIONS:
        assert abs(K.ratio_closed_form(conv, mp.mpf("0.5") * C, mp.mpf(0)) - 1) < TOL


def test_ratio_decreases_toward_the_body():
    """안쪽으로 갈수록 시계가 느려져야 한다 (단조성)."""
    GM = mp.mpf(10) * GM_SUN
    rs = 2 * GM / C**2
    ratios = [model.time_ratio(f * rs, GM) for f in (1.5, 2, 5, 20, 100)]
    assert all(a < b for a, b in zip(ratios, ratios[1:]))


def test_horizon_is_where_flow_reaches_c():
    GM = mp.mpf(3) * GM_SUN
    assert abs(model.flow_speed(model.horizon_radius(GM), GM) - C) < TOL


def test_convention_A_inversion_raises_at_u_equals_c():
    with pytest.raises(K.DegenerateInversion):
        K.solve_w("A", mp.mpf("0.5"), C)


def test_convention_B2_has_no_solution_at_u_equals_c():
    with pytest.raises(K.DegenerateInversion):
        K.solve_w("B2", mp.mpf("0.5"), C)


def test_unknown_convention_rejected():
    with pytest.raises(ValueError):
        K.ratio_closed_form("B3", C, mp.mpf(0))


In [ ]:
import sys



sys.path.insert(0, str(PKG_DIR))

print('준비 완료 -- 저장소 클론 없이 완전히 독립적으로 실행됨')

## 3. 모델 불러오기

`flowline` 패키지 구성:

| 모듈 | 내용 |
|---|---|
| `constants` | 물리 상수, 가상 천체 목록 (`BODIES`) |
| `kinematics` | 1차원 구성의 순수 특수상대론 부분 — Lorentz 부스트, 측정 규약 A/B1/B2/C |
| `theory` | 비교 대상: Schwarzschild, Newton |
| `model` | 역방향 교정 / 순방향 예측 — `w(r)`, `Φ`, `g`, 지평선, 진공 닫힘 조건 |
| `checks` | 정합성 검사 10종 |
| `report` | 비교표 3종 |
| `plots` | 그림 3종 |

In [ ]:
from flowline import checks, model, plots, report, theory

from flowline.constants import BODIES

from mpmath import mp



print(f'계산 정밀도: {mp.dps} 자리 (mpmath)')

print(f'가상 천체 {len(BODIES)}개:', ', '.join(b.label for b in BODIES))

## 4. 비교표

핵심 결과: `w(r) = √(2GM/r)` (그 지점의 탈출속도) 로 놓으면 `√(1 - w²/c²)` 가 Schwarzschild 의 `√(1 - r_s/r)` 와 **근사가 아니라 항등**으로 일치한다. 아래 표의 '모델-Schw (상대)' 열이 그 잔차인데, 배정밀도 잡음 수준(약 `1e-16`)보다 한참 작은 `1e-60` 대까지 내려간다 — 그래서 이 저장소는 float 대신 mpmath 60자리로 계산한다.

In [ ]:
print(report.table_time_dilation())

`Φ = -w²/2`, `g = w·dw/dr` 가 Newton 값과 일치하는지. `g` 는 해석해를 대입한 것이 아니라 흐름장 `w(r)` 를 **수치 미분**해 얻었다 — '중력장 = 흐름의 이류 가속도' 가 실제로 성립함을 보이기 위해서다.

In [ ]:
print(report.table_potential_and_field())

측정 규약 A/B1/B2/C 가 왜 갈리는지를 한 지점(중성자별)에서 역산으로 보여준다. B1 만 점의 속력 `u` 에 무관하면서 탈출속도와 정확히 일치한다.

In [ ]:
print(report.table_conventions())

## 5. 정합성 검사

In [ ]:
n_fail = 0

for name, ok, detail in checks.run_all():

    n_fail += not ok

    print(('PASS' if ok else 'FAIL'), '|', name)

    print('       ', detail)

print(f'\n총 {len(checks.ALL_CHECKS)}개 중 실패 {n_fail}개.')

assert n_fail == 0

## 6. 그림

- **fig1** — `t(r)/t(∞)`, `w/c`, `Φ`, `g` 의 반경 프로파일. 모델(실선)과 기존 이론(파선)이 겹쳐 하나로 보인다.
- **fig2** — 측정 규약 선택의 근거: B2 의 Doppler 오염, A 의 `u` 의존성.
- **fig3** — 약한장 잔차가 `(1/8)(r_s/r)²` 로, 즉 2차 항 계수까지 정확히 사라지는지.

In [ ]:
from pathlib import Path

from IPython.display import Image, display



figdir = Path('/content/figures')

figdir.mkdir(exist_ok=True)



plots.figure_profiles(figdir / 'fig1_profiles.png')

plots.figure_conventions(figdir / 'fig2_conventions.png')

plots.figure_weak_field(figdir / 'fig3_weak_field.png')



for name in ('fig1_profiles.png', 'fig2_conventions.png', 'fig3_weak_field.png'):

    display(Image(filename=str(figdir / name)))

## 7. (선택) 테스트 스위트

pytest 34개 — 부스트 유도와 닫힌 형태의 일치, `u` 독립성, 역산 왕복, 단조성, 지평선 등을 검사한다.

In [ ]:
!pip install -q pytest

!cd /content/pkg && python -m pytest tests/ -q

## 한계

1. 횡방향/궤도 운동이 없다 — 1차원 반경 슬라이스만 담으므로 빛의 휘어짐, 근일점 이동은 나오지 않는다.
2. 선호 기준계(에테르) 그림이다 — 국소적 검출 불가능성은 따로 보여야 한다.
3. 회전 천체(Kerr) 에는 비틀림이 있는 흐름이 필요하다.
4. "거리비 = 시간비" 는 정의이지 유도가 아니다.
5. `r < r_s` 에서 `w > c` — 공간 흐름은 좌표 효과라 가능하지만 신호는 불가.